# ⏱️ RS-LiDAR & Vanilla LiDAR: Benchmark Table 2 Efficiency (Time & Peak VRAM on A100)

> **Mục tiêu**: Đo đạc độc lập, chuẩn xác thời gian thực thi phân rã 3 thành phần (khớp Bảng 9) và đỉnh VRAM (Bảng 2) của **RS-LiDAR** và **Vanilla LiDAR** trên GPU **NVIDIA A100**:
> 1. **$T_{\text{lookahead}}$**: Thời gian sinh $N$ hạt lookahead (DPM-5 hoặc DMD-1) và giải mã VAE.
> 2. **$T_{\text{reward}}$**: Thời gian đánh giá ImageReward (**Full-Batch 1 forward duy nhất** trên trọn vẹn $M \times N$ ảnh).
> 3. **$T_{\text{target}}$**: Thời gian khử nhiễu mục tiêu (Closed-form steering, sinh 4 ảnh đích, **DỪNG NGAY KHÔNG CHẤM REWARD Ở PHASE 2**).
> 4. **$T_{\text{total}}$**: $T_{\text{lookahead}} + T_{\text{reward}} + T_{\text{target}}$.
> 5. **Peak VRAM**: $\text{torch.cuda.max\_memory\_allocated()} / 1024^3$ (GiB).
> 6. **Auto Google Drive Sync**: Tự động lưu và đồng bộ tức thì file kết quả CSV & JSON về Google Drive cá nhân sau từng prompt.

---

### 📌 3 Cấu Hình Benchmark Chuẩn Bảng 2:
| Setting Key | Model | Lookahead Solver | Target Solver | Scale | $\eta$ |
| :--- | :--- | :--- | :--- | :---: | :---: |
| `sdv1.5_ddim50` | SD v1.5 | DPM-5 ($N=50$) | DDIM 50 steps ($N=4$) | 12.5 | 0.0 |
| `sdv1.5_ddpm100` | SD v1.5 | DPM-5 ($N=50$) | DDPM 100 steps ($N=4$) | 12.5 | 1.0 |
| `sdxl_dmd1` | SDXL (2.6B) | DMD-1 ($N=100$, 1 bước) | DDPM 100 steps ($N=4$) | 8.0 | 1.0 |

## 1. 🖥️ Kiểm Tra Phần Cứng GPU (Khuyến nghị NVIDIA A100)

In [ ]:
!nvidia-smi

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"\n✅ GPU khả dụng: {gpu_name} ({total_vram:.2f} GiB VRAM)")
    if "A100" in gpu_name:
        print("🌟 Bạn đang chạy trên NVIDIA A100 chuẩn benchmark bài báo ICML 2026!")
    else:
        print("ℹ️ Lưu ý: GPU hiện tại không phải A100. Thời gian thực tế có thể chênh lệch so với Bảng 2 bài báo.")
else:
    print("❌ Không tìm thấy GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn GPU (A100).")

## 2. 📁 Gắn Kết Google Drive Cá Nhân (Tự Động Lưu Kết Quả Chống Mất Dữ Liệu)

In [ ]:
# @title 📁 Gắn kết Google Drive & Thiết lập thư mục lưu trữ
import os
from google.colab import drive

MOUNT_DRIVE = True # @param {type:"boolean"}
DRIVE_BENCHMARK_DIR = None

if MOUNT_DRIVE:
    try:
        drive.mount('/content/drive')
        base_drive = None
        for cand in ['/content/drive/MyDrive', '/content/drive/My Drive', '/content/drive']:
            if os.path.exists(cand):
                base_drive = cand
                break
        if base_drive is None:
            base_drive = '/content/drive/MyDrive'

        DRIVE_BENCHMARK_DIR = os.path.join(base_drive, "RS-LiDAR", "results", "benchmark_efficiency")
        os.makedirs(DRIVE_BENCHMARK_DIR, exist_ok=True)
        print(f"\n✅ Đã gắn kết Google Drive thành công!")
        print(f"📁 Thư mục tự động sao lưu trên Drive: {DRIVE_BENCHMARK_DIR}")
    except Exception as e:
        print(f"⚠️ Không thể mount Google Drive: {e}. Kết quả sẽ chỉ được lưu tạm thời trên đĩa Colab.")
        DRIVE_BENCHMARK_DIR = None
else:
    print("ℹ️ Bỏ qua gắn kết Google Drive. Kết quả sẽ lưu cục bộ tại 'results/benchmark_efficiency/'.")

## 3. 📂 Đồng Bộ Mã Nguồn Từ GitHub & Định Tuyến Thư Mục

In [ ]:
# @title 📥 Bước 3: Clone / Update Repository RS-LiDAR và Chuyển Thư Mục
import os

!git clone https://github.com/leekwanreal/RS-LiDAR.git /content/RS-LiDAR || git -C /content/RS-LiDAR pull origin main
%cd /content/RS-LiDAR

assert os.path.exists("benchmark_table2_efficiency.py"), f"❌ Chưa tìm thấy benchmark_table2_efficiency.py trong {os.getcwd()}!"
print(f"\n✅ Đã ở đúng thư mục làm việc: {os.getcwd()}")


## 4. 📦 Cài Đặt Thư Viện Cần Thiết

In [ ]:
# 1. Tat backend TensorFlow de loai bo xung dot Protobuf runtime_version
import os, urllib.request
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'

# 2. Nang cap protobuf va cai dat cac thu vien tuong thich PyTorch 2.x chuan Table 2 Replication
!pip install -q --upgrade protobuf open_clip_torch
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q matplotlib tqdm scipy seaborn pandas tabulate

print('\n✅ Cai dat moi truong chuan xac 100% tu Table 2 Replication (Khong xung dot Protobuf)!')


## 5. ⚙️ Cấu Hình Thực Nghiệm (Interactive Form)

In [ ]:
# @title 🎛️ Chọn Tham Số Benchmark

# @markdown **0. Chế độ thực thi khi bấm Run All (Execution Mode)**:
EXECUTION_MODE = "sweep_steps" # @param ["sweep_steps", "single_step", "compare_both"]
# @markdown *(Khuyến nghị: 'sweep_steps' để chạy liên tiếp 2, 3, 4, 5 steps ở Cell 9 nạp model 1 lần duy nhất; 'single_step' chỉ chạy 1 mức bước ở Cell 6; 'compare_both' chạy so sánh ở Cell 8)*

# @markdown **1. Cấu hình bài báo (Setting)**:
SETTING = "sdv1.5_ddpm100" # @param ["sdv1.5_ddim50", "sdv1.5_ddpm100", "sdxl_dmd1"]

# @markdown **2. Phương pháp lấy mẫu (Method)**:
METHOD = "rs-lidar" # @param ["rs-lidar", "lidar"]

# @markdown **3. Số bước sinh ảnh Phase 1 (Lookahead Steps)**:
LOOKAHEAD_STEPS = 5 # @param {type:"integer"}
# @markdown *(Gợi ý: 2, 3, 4, 5 cho SD 1.5 DPM-Solver; Tự động về 1 cho SDXL DMD-1. Mặc định: 5)*

# @markdown **4. Cơ chế chọn prompt (Prompt Selection Strategy)**:
PROMPT_SELECTION_MODE = "stratified_by_category" # @param ["stratified_by_category", "first_n"]
# @markdown *(Khuyến nghị: 'stratified_by_category' chọn đều các loại task để chuẩn xác khoa học)*

# @markdown **5. Số prompt cho mỗi category (nếu chọn stratified)**:
PROMPTS_PER_CATEGORY = 2 # @param {type:"integer"}
# @markdown *(Mặc định: 2 prompt x 6 categories = 12 prompts bao quát đủ: Single Object, Two Object, Counting, Colors, Position, Color Attribute)*

# @markdown **6. Số lượng prompts (nếu chọn first_n)**:
NUM_PROMPTS = 3 # @param {type:"integer"}

# @markdown **7. Độ lệch chuẩn nhiễu làm mịn (Sigma)**:
SIGMA = 1.0 # @param {type:"number"}
# @markdown *(Gợi ý: 1.0 cho SD 1.5; Tự động về 0.5 cho SDXL; Tự động về 0.0 nếu chọn Vanilla LiDAR)*

# @markdown **8. Số mẫu Monte Carlo (M)**:
NUM_MC_SAMPLES = 4 # @param {type:"integer"}
# @markdown *(Gợi ý: 4 cho RS-LiDAR; Tự động về 1 nếu chọn Vanilla LiDAR)*

# @markdown **9. Batch Size ImageReward (Mặc định: Full Batch)**:
REWARD_BATCH_SIZE = "None" # @param ["None", "64", "32", "16"]
# @markdown *(Giữ 'None' để ép chạy trọn vẹn 1 batch duy nhất trên A100!)*

# @markdown **10. Bật Warm-up GPU (1 prompt khởi động kernel)**:
ENABLE_WARMUP = True # @param {type:"boolean"}

# @markdown **11. Tự động sao lưu kết quả sang Google Drive**:
SAVE_TO_DRIVE = True # @param {type:"boolean"}

# Auto-Dispatch thông số cho SDXL DMD-1
is_sdxl = "sdxl" in SETTING
effective_lookahead_steps = 1 if is_sdxl else LOOKAHEAD_STEPS
effective_sigma = 0.5 if (is_sdxl and METHOD == "rs-lidar") else (SIGMA if METHOD == "rs-lidar" else 0.0)

print(f"Cấu hình đã chọn:")
print(f"• Execution Mode:    {EXECUTION_MODE}")
print(f"• Setting:           {SETTING}")
print(f"• Method:            {METHOD.upper()}")
if is_sdxl:
    print(f"• Lookahead Steps:   1 (⚡ Auto-Dispatch: SDXL DMD-1 là mô hình chưng cất 1-step)")
    print(f"• Smoothing Sigma:   {effective_sigma} (⚡ Auto-Dispatch: Chuẩn Table 2 bài báo)")
else:
    print(f"• Lookahead Steps:   {effective_lookahead_steps}")
    print(f"• Smoothing Sigma:   {effective_sigma}")
if PROMPT_SELECTION_MODE == "stratified_by_category":
    print(f"• Prompt Selection:  Phân tầng {PROMPTS_PER_CATEGORY} prompts/category (Tổng: {PROMPTS_PER_CATEGORY * 6} prompts)")
else:
    print(f"• Prompt Selection:  Lấy {NUM_PROMPTS} prompts đầu")
print(f"• Monte Carlo M:     {NUM_MC_SAMPLES if METHOD == 'rs-lidar' else 1}")
print(f"• Reward Batch Size: {REWARD_BATCH_SIZE}")
print(f"• GPU Warmup:        {ENABLE_WARMUP}")
print(f"• Save to Drive:     {SAVE_TO_DRIVE and DRIVE_BENCHMARK_DIR is not None}")

## 6. 🚀 Thực Thi Benchmark Table 2 Efficiency

In [ ]:
import os

if EXECUTION_MODE == "single_step":
    # Đảm bảo đứng đúng thư mục chứa benchmark script
    if not os.path.exists("benchmark_table2_efficiency.py"):
        %cd /content/RS-LiDAR

    prompt_flags = ["--prompts_per_category", str(PROMPTS_PER_CATEGORY)] if PROMPT_SELECTION_MODE == "stratified_by_category" else ["--num_prompts", str(NUM_PROMPTS)]
    warmup_flag = "--warmup" if ENABLE_WARMUP else "--no_warmup"
    cmd = [
        "python", "benchmark_table2_efficiency.py",
        "--setting", SETTING,
        "--method", METHOD,
        *prompt_flags,
        "--lookahead_steps", str(effective_lookahead_steps),
        "--sigma", str(effective_sigma),
        "--num_mc_samples", str(NUM_MC_SAMPLES),
        "--reward_batch_size", str(REWARD_BATCH_SIZE),
        warmup_flag,
        "--output_dir", "results/benchmark_efficiency"
    ]

    # Thêm cờ đồng bộ Google Drive nếu có
    if SAVE_TO_DRIVE and DRIVE_BENCHMARK_DIR is not None:
        cmd.extend(["--drive_backup_dir", f"'{DRIVE_BENCHMARK_DIR}'"])

    print(f"Chạy lệnh: {' '.join(cmd)}\n")
    !{' '.join(cmd)}
else:
    print(f"ℹ️ Bỏ qua Cell 6 (Chế độ hiện tại: '{EXECUTION_MODE}'). Cell tương ứng sẽ tự động chạy bên dưới!")


## 7. 📊 Hiển Thị Bảng Phân Rã Thời Gian (Khớp Table 9 & Table 2 Bài Báo Gốc)

In [ ]:
import os, glob, json
import pandas as pd
from tabulate import tabulate
from IPython.display import display, HTML

if EXECUTION_MODE == "sweep_steps":
    print("ℹ️ Đang chạy chế độ 'sweep_steps'. Bảng tổng hợp Ablation Study chi tiết sẽ được tự động hiển thị tại Cell 9 bên dưới.")
else:
    
    # Baseline tham chiếu chính thức từ bài báo gốc ICML 2026 (Table 9 & Table 2 / Figure 10)
    PAPER_BASELINES = {
        "sdv1.5_ddim50": {
            "name": "SD 1.5 (DDIM-50 / n=50)",
            "t_lookahead": 5.69,
            "t_reward": 0.65,
            "t_target": 3.58,
            "t_total": 9.92,
            "vram_phase2": 8.90,
        },
        "sdv1.5_ddpm100": {
            "name": "SD 1.5 (DDPM-100 / n=50)",
            "t_lookahead": 5.69,
            "t_reward": 0.65,
            "t_target": 7.07,
            "t_total": 13.41,
            "vram_phase2": 8.90,
        },
        "sdxl_dmd1": {
            "name": "SDXL (DMD-1 / n=100)",
            "t_lookahead": 4.30,
            "t_reward": 1.30,
            "t_target": 50.00,
            "t_total": 55.60,
            "vram_phase2": 33.84,
        },
    }
    
    # Ưu tiên đọc từ Google Drive nếu có sao lưu, fallback về local
    search_dirs = []
    if DRIVE_BENCHMARK_DIR and os.path.exists(DRIVE_BENCHMARK_DIR):
        search_dirs.append(DRIVE_BENCHMARK_DIR)
    search_dirs.append("results/benchmark_efficiency")
    
    json_files = []
    for s_dir in search_dirs:
        json_files.extend(glob.glob(os.path.join(s_dir, f"efficiency_{SETTING}_{METHOD}_*.json")))
    if not json_files:
        for s_dir in search_dirs:
            json_files.extend(glob.glob(os.path.join(s_dir, "*.json")))
    
    if json_files:
        latest_json = max(json_files, key=os.path.getmtime)
        with open(latest_json, "r", encoding="utf-8") as f:
            data = json.load(f)
    
        ref = PAPER_BASELINES.get(SETTING, {})
        measured_step = data.get('lookahead_steps', globals().get('LOOKAHEAD_STEPS', 5))
        step_label = f" (DPM-{measured_step})" if "sdv1.5" in SETTING else f" (DMD-{measured_step})"
        vram_p2_val = data.get('mean_vram_phase2_gib', data.get('max_peak_vram_gib', 0.0))
        conv_sec = data.get('mean_t_convert_sec', 0.0)
        conv_str = f" (+{conv_sec:.2f}s conv)" if conv_sec > 0.0 else ""
    
        rows = [
            [
                f"Vanilla LiDAR (Paper)",
                f"{ref.get('t_lookahead', '-'):.2f}s",
                f"{ref.get('t_reward', '-'):.2f}s",
                f"{ref.get('t_target', '-'):.2f}s",
                f"{ref.get('t_total', '-'):.2f}s",
                f"{ref.get('vram_phase2', '-'):.2f} GiB",
                f"{ref.get('vram_phase2', '-'):.2f} GiB",
            ],
            [
                f"<b>{METHOD.upper()} (Measured{step_label})</b>",
                f"<b>{data['mean_t_lookahead_sec']:.2f}s</b>",
                f"<b>{data['mean_t_reward_sec']:.2f}s{conv_str}</b>",
                f"<b>{data['mean_t_target_sec']:.2f}s</b>",
                f"<b>{data['mean_t_total_sec']:.2f}s</b>",
                f"<b>{vram_p2_val:.2f} GiB</b>",
                f"<b>{data['max_peak_vram_gib']:.2f} GiB</b>",
            ]
        ]
    
        headers = ["Method", "T_lookahead", "T_reward (Scoring)", "T_target", "T_total (Table 9)", "VRAM P2 (Target)", "Peak Overall VRAM"]
        df_summary = pd.DataFrame(rows, columns=headers)
    
        print(f"\n{'='*95}")
        print(f"📊 KẾT QUẢ ĐỐI CHIẾU TABLE 9 & TABLE 2 (Setting: {SETTING})")
        print(f"{'='*95}")
        print(tabulate(rows, headers=headers, tablefmt="grid"))
        print(f"{'='*95}\n")
    
        # Hiển thị bảng HTML rich
        display(HTML(f"<h3>📊 Bảng Đối Chiếu Thời Gian & Bộ Nhớ: {ref.get('name', SETTING)}</h3>"))
        display(HTML(df_summary.to_html(escape=False, index=False)))
        print("\n💡 Ghi chú khoa học:")
        print("  • T_lookahead: Sinh 50 hạt DPM-5 + VAE decode ra ảnh PIL (Bài báo Table 9: 5.69s).")
        print("  • T_reward (Scoring): Thời gian chấm điểm ImageReward thuần túy qua mạng ViT+BERT (không tính khâu đổi tensor->PIL).")
        print("  • VRAM P2: Đỉnh bộ nhớ riêng biệt của Phase 2 Target Sampling khớp 8.90 GiB theo Table 2 & Figure 10 bài báo.")
        if DRIVE_BENCHMARK_DIR:
            print(f"\n☁️ Kết quả đã được lưu vĩnh viễn trên Google Drive: {DRIVE_BENCHMARK_DIR}")
    else:
        print("⚠️ Chưa tìm thấy file kết quả trong 'results/benchmark_efficiency/'.")

## 8. 🔄 Chạy Tự Động So Sánh Trực Tiếp Cả 2 (Vanilla LiDAR vs RS-LiDAR)
> Cell này sẽ tự động chạy liên tiếp cả **Vanilla LiDAR** và **RS-LiDAR** trên cùng setting và đồng bộ cả 2 về Google Drive.

In [ ]:
# @title ⚡ Auto Comparative Run (LiDAR vs RS-LiDAR)
RUN_COMPARISON = False # @param {type:"boolean"}

if EXECUTION_MODE == "compare_both" or RUN_COMPARISON:
    if not os.path.exists("benchmark_table2_efficiency.py"):
        %cd /content/RS-LiDAR

    print("\n🚀 BẮT ĐẦU CHẠY SO SÁNH 2 PHƯƠNG PHÁP...")
    drive_opt = f"--drive_backup_dir '{DRIVE_BENCHMARK_DIR}'" if (SAVE_TO_DRIVE and DRIVE_BENCHMARK_DIR) else ""
    step_opt = f"--lookahead_steps {effective_lookahead_steps}"
    prompt_opt = f"--prompts_per_category {PROMPTS_PER_CATEGORY}" if PROMPT_SELECTION_MODE == "stratified_by_category" else f"--num_prompts {NUM_PROMPTS}"
    
    # 1. Chạy Vanilla LiDAR
    print("\n--- [1/2] Đang chạy Vanilla LiDAR ---")
    !python benchmark_table2_efficiency.py --setting {SETTING} --method lidar {prompt_opt} {step_opt} --reward_batch_size None --warmup {drive_opt}
    
    # 2. Chạy RS-LiDAR
    print("\n--- [2/2] Đang chạy RS-LiDAR ---")
    !python benchmark_table2_efficiency.py --setting {SETTING} --method rs-lidar {prompt_opt} {step_opt} --sigma {effective_sigma} --num_mc_samples {NUM_MC_SAMPLES} --reward_batch_size None --warmup {drive_opt}
    
    print("\n✅ Hoàn thành so sánh cả 2 phương pháp! Chạy lại Cell 7 để xem bảng tổng hợp đối chiếu.")
else:
    print(f"ℹ️ Bỏ qua Cell 8 (Chế độ hiện tại: '{EXECUTION_MODE}'). Đặt EXECUTION_MODE = 'compare_both' để kích hoạt.")


## 9. ⚡ Chạy Nhanh Sweep Nhiều Bước Phase 1 ($\delta \in \{2, 3, 4, 5\}$ Steps) - Đo Phân Rã Thời Gian
> Cell này sẽ tự động chạy liên tiếp các số bước Phase 1 (ví dụ `2, 3, 4, 5`) **trong cùng 1 phiên nạp model** (nạp model 1 lần duy nhất để tối ưu thời gian đo).
> Kết quả sẽ tự động được tổng hợp thành bảng đối chiếu chi tiết tương ứng với Bảng **Ablation Study 2 (Lookahead Steps $\delta \in \{2, 3, 4, 5\}$)** trong bài báo.

In [ ]:
# @title ⚡ Fast Multi-Step Sweep Benchmark (2, 3, 4, 5 Steps)
SWEEP_STEPS = "2, 3, 4, 5" # @param {type:"string"}
RUN_SWEEP = True # @param {type:"boolean"}

if EXECUTION_MODE == "sweep_steps" or (EXECUTION_MODE not in ["single_step", "compare_both"] and RUN_SWEEP):
    import os, glob, json
    import pandas as pd
    from tabulate import tabulate
    from IPython.display import display, HTML

    if not os.path.exists("benchmark_table2_efficiency.py"):
        %cd /content/RS-LiDAR

    drive_opt = f"--drive_backup_dir '{DRIVE_BENCHMARK_DIR}'" if (SAVE_TO_DRIVE and DRIVE_BENCHMARK_DIR) else ""
    prompt_flags = ["--prompts_per_category", str(PROMPTS_PER_CATEGORY)] if PROMPT_SELECTION_MODE == "stratified_by_category" else ["--num_prompts", str(NUM_PROMPTS)]
    warmup_flag = "--warmup" if ENABLE_WARMUP else "--no_warmup"

    if is_sdxl:
        print("\n⚡ [SDXL DMD-1 Auto-Dispatch]: Phát hiện cấu hình SDXL DMD-1!")
        print("   • DMD-1 là mô hình chưng cất 1-step (Table 2), tự động bỏ sweep đa bước.")
        print("   • Chỉ chạy 1 lần duy nhất với Lookahead Steps = 1.\n")
        cmd = [
            "python", "benchmark_table2_efficiency.py",
            "--setting", SETTING,
            "--method", METHOD,
            *prompt_flags,
            "--lookahead_steps", "1",
            "--sigma", str(effective_sigma),
            "--num_mc_samples", str(NUM_MC_SAMPLES),
            "--reward_batch_size", str(REWARD_BATCH_SIZE),
            warmup_flag,
            "--output_dir", "results/benchmark_efficiency"
        ]
    else:
        print(f"\n🚀 BẮT ĐẦU CHẠY SWEEP CÁC BƯỚC PHASE 1: [{SWEEP_STEPS}]...")
        cmd = [
            "python", "benchmark_table2_efficiency.py",
            "--setting", SETTING,
            "--method", METHOD,
            *prompt_flags,
            "--sweep_steps", f'"{SWEEP_STEPS}"',
            "--sigma", str(effective_sigma),
            "--num_mc_samples", str(NUM_MC_SAMPLES),
            "--reward_batch_size", str(REWARD_BATCH_SIZE),
            warmup_flag,
            "--output_dir", "results/benchmark_efficiency"
        ]
    if SAVE_TO_DRIVE and DRIVE_BENCHMARK_DIR is not None:
        cmd.extend(["--drive_backup_dir", f"'{DRIVE_BENCHMARK_DIR}'"])

    print(f"Chạy lệnh: {' '.join(cmd)}\n")
    !{' '.join(cmd)}

    # Hiển thị bảng tổng hợp kết quả
    search_dirs = []
    if DRIVE_BENCHMARK_DIR and os.path.exists(DRIVE_BENCHMARK_DIR):
        search_dirs.append(DRIVE_BENCHMARK_DIR)
    search_dirs.append("results/benchmark_efficiency")

    if is_sdxl:
        # Với SDXL: Đọc file kết quả đơn 1-step và hiển thị đối chiếu Table 2
        json_files = []
        for s_dir in search_dirs:
            json_files.extend(glob.glob(os.path.join(s_dir, f"efficiency_{SETTING}_{METHOD}_*.json")))
        if json_files:
            latest_json = max(json_files, key=os.path.getmtime)
            with open(latest_json, "r", encoding="utf-8") as f:
                data = json.load(f)
            ref_rows = [
                ["Vanilla LiDAR (Paper DMD-1)", "4.30s", "1.30s", "50.00s", "55.60s", "33.84 GiB", "33.84 GiB"],
                [
                    f"<b>{METHOD.upper()} (Measured DMD-1)</b>",
                    f"<b>{data['mean_t_lookahead_sec']:.2f}s</b>",
                    f"<b>{data['mean_t_reward_sec']:.2f}s</b>",
                    f"<b>{data['mean_t_target_sec']:.2f}s</b>",
                    f"<b>{data['mean_t_total_sec']:.2f}s</b>",
                    f"<b>{data.get('mean_vram_phase2_gib', data.get('max_peak_vram_gib', 0.0)):.2f} GiB</b>",
                    f"<b>{data['max_peak_vram_gib']:.2f} GiB</b>",
                ]
            ]
            headers = ["Method", "T_lookahead", "T_reward", "T_target", "T_total (Table 9)", "VRAM P2", "Peak VRAM"]
            print(f"\n{'='*95}")
            print(f"📊 KẾT QUẢ ĐỐI CHIẾU TABLE 2 (SDXL DMD-1 / 1-Step)")
            print(f"{'='*95}")
            print(tabulate(ref_rows, headers=headers, tablefmt="grid"))
            print(f"{'='*95}\n")
            display(HTML(f"<h3>📊 Bảng Đối Chiếu SDXL DMD-1 (1-Step Table 2)</h3>"))
            display(HTML(pd.DataFrame(ref_rows, columns=headers).to_html(escape=False, index=False)))
    else:
        # Với SD 1.5: Đọc file sweep đa bước
        sweep_files = []
        for s_dir in search_dirs:
            sweep_files.extend(glob.glob(os.path.join(s_dir, f"efficiency_sweep_{SETTING}_{METHOD}_*.json")))
        if sweep_files:
            latest_sweep_json = max(sweep_files, key=os.path.getmtime)
            with open(latest_sweep_json, "r", encoding="utf-8") as f:
                sweep_data = json.load(f)
            sweep_rows = []
            for sm in sweep_data.get("sweep_results", []):
                st = sm.get("lookahead_steps")
                st_name = f"δ = {st} (DPM-{st})"
                conv_sec = sm.get("mean_t_convert_sec", 0.0)
                conv_str = f" (+{conv_sec:.2f}s conv)" if conv_sec > 0.0 else ""
                sweep_rows.append([
                    st_name,
                    f"{sm.get('mean_t_lookahead_sec', 0.0):.2f}s",
                    f"{sm.get('mean_t_reward_sec', 0.0):.2f}s{conv_str}",
                    f"{sm.get('mean_t_target_sec', 0.0):.2f}s",
                    f"{sm.get('mean_t_total_sec', 0.0):.2f}s",
                    f"{sm.get('mean_vram_phase2_gib', 0.0):.2f} GiB",
                    f"{sm.get('max_peak_vram_gib', 0.0):.2f} GiB"
                ])
            headers = ["Lookahead Steps (δ)", "T_lookahead", "T_reward (Scoring)", "T_target", "T_total (Table 9)", "VRAM P2", "Peak VRAM"]
            print(f"\n{'='*95}")
            print(f"📊 BẢNG TỔNG HỢP ABLATION LOOKAHEAD STEPS (Setting: {SETTING} | Method: {METHOD.upper()})")
            print(f"{'='*95}")
            print(tabulate(sweep_rows, headers=headers, tablefmt="grid"))
            print(f"{'='*95}\n")
            display(HTML(f"<h3>📊 Bảng Tổng Hợp Ablation: Lookahead Steps $\\delta \\in \\{{{SWEEP_STEPS}\\}}$</h3>"))
            df_sweep = pd.DataFrame(sweep_rows, columns=headers)
            display(HTML(df_sweep.to_html(escape=False, index=False)))
else:
    print(f"ℹ️ Bỏ qua Cell 9 (Chế độ hiện tại: '{EXECUTION_MODE}'). Đặt EXECUTION_MODE = 'sweep_steps' để kích hoạt.")
